In [2]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
import os
import math
from datetime import datetime
from langchain_core.tools import tool
from langchain.agents import create_agent
 
load_dotenv(override=True)
 
#print(os.environ.get("ANTHROPIC_API_KEY")[:3])  # Check if it's set
 
# Initialize the Claude model
llm = ChatAnthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    model=os.environ.get("CLAUDE_MODEL"),             # Required: model ID to use
    temperature=1.0,                       # Randomness: 0.0 = deterministic, 1.0 = default (max varies by model)
    max_tokens=4096,                       # Max tokens in response (default from model profile, fallback 4096)
    top_k=None,                            # Top-K sampling: consider only top K tokens (None = disabled)
    top_p=None,                            # Top-P sampling: nucleus sampling threshold (None = disabled)
    timeout=None,                          # Request timeout in seconds (None = no timeout)
    max_retries=2,                         # Number of retries on failed requests
    stop=None,                             # Stop sequences e.g. ["\n", "END"] (None = disabled)
    #base_url="https://api.anthropic.com",  # API base URL (can also set via ANTHROPIC_API_URL env var)
    streaming=True,                       # Stream tokens as generated (False = wait for full response)
    thinking=None,
    base_url=os.environ.get("ENDPOINT")                                          # Extended thinking e.g. {"type": "enabled", "budget_tokens": 5000}
)

/home/ubuntu/My_learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Supports basic operations (+, -, *, /, **) and functions like sqrt, sin, cos.
    Examples: '2 + 2', '10 ** 2', 'sqrt(16)', '100 / 4'"""
    try:
        # Safe evaluation with limited functions
        allowed_names = {
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos,
            'tan': math.tan, 'log': math.log, 'pi': math.pi, 'e': math.e
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_datetime() -> str:
    """Get the current date and time. Use this when the user asks about today's date or current time."""
    now = datetime.now()
    return f"Current date and time: {now.strftime('%A, %B %d, %Y at %I:%M %p')}"

@tool
def search_knowledge_base(query: str) -> str:
    """Search our internal knowledge base for information about AI, Machine Learning, and LangChain."""
    # Simulated knowledge base
    knowledge = {
        "langchain": "LangChain is an open-source framework for building LLM applications. Created by Harrison Chase in 2022. Key components: Chains, Agents, Memory, Tools.",
        "agentic ai": "Agentic AI refers to AI systems that can plan, reason, and take autonomous actions to achieve goals. Key features: tool use, memory, multi-step reasoning.",
        "claude": "Claude is an AI assistant created by Anthropic. Known for being helpful, harmless, and honest. Supports up to 200K context window.",
        "react": "ReAct (Reason + Act) is a prompting pattern where the AI thinks step-by-step, takes actions, observes results, and repeats until done.",
        "rag": "RAG (Retrieval-Augmented Generation) combines LLMs with external knowledge retrieval. Steps: Query → Retrieve documents → Generate answer with context."
    }
    
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return f"Knowledge Base Result:\n{value}"
    return "No matching information found in the knowledge base."

# Collect all tools
tools = [calculator, get_current_datetime, search_knowledge_base]
print("Tools defined:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

Tools defined:
  - calculator: Evaluate a mathematical expression. Supports basic operation...
  - get_current_datetime: Get the current date and time. Use this when the user asks a...
  - search_knowledge_base: Search our internal knowledge base for information about AI,...


In [9]:
system_prompt = """You are a helpful research assistant with access to tools.

Your capabilities:
- Calculate mathematical expressions
- Get the current date and time
- Search our knowledge base about AI topics

Guidelines:
1. Always use tools when you need factual information
2. Think step-by-step before answering complex questions
3. Cite your sources when using the knowledge base
4. If you don't know something, say so honestly
"""

print("System prompt created!")

System prompt created!


In [12]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Agent created!")

Agent created!


In [13]:
chat_history = []  # Our memory store

def chat(user_input: str) -> str:
    """Send a message to the agent and update memory."""
    global chat_history
    
    # Build messages list: chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Chat function ready!")

Chat function ready!


In [14]:
print("="*60)
print("Test 1: Math calculation")
print("="*60)
response = chat("What is 15% of 250?")
print(f"\nFinal Answer: {response}")

Test 1: Math calculation

Final Answer: [{'text': '15% of 250 is **37.5**.', 'type': 'text', 'index': 0}]


In [15]:
print("="*60)
print("Test 2: Knowledge lookup")
print("="*60)
response = chat("What is LangChain and who created it?")
print(f"\nFinal Answer: {response}")

Test 2: Knowledge lookup

Final Answer: [{'text': 'Based on the knowledge base:\n\n**LangChain** is an open-source framework designed for building applications powered by Large Language Models (LLMs).\n\n**Creator:** LangChain was created by **Harrison Chase** in **2022**.\n\n**Key Components:**\n- **Chains** - sequences of operations for processing\n- **Agents** - autonomous decision-making components\n- **Memory** - for maintaining context and history\n- **Tools** - integrations with external services and APIs\n\nLangChain provides developers with tools and abstractions to make it easier to work with LLMs and build complex applications that leverage language model capabilities.', 'type': 'text', 'index': 0}]


In [16]:
print("="*60)
print("Test 3: Complex question requiring multiple tools")
print("="*60)
response = chat("What is today's date, and if LangChain was created in 2022, how many years ago was that?")
print(f"\nFinal Answer: {response}")

Test 3: Complex question requiring multiple tools

Final Answer: [{'text': "Today's date is **Saturday, June 27, 2026** at 10:03 AM.\n\nLangChain was created in 2022, which means it was created **4 years ago** (2026 - 2022 = 4 years).", 'type': 'text', 'index': 0}]


In [17]:
print("="*60)
print("Test 4: Memory - follow-up question")
print("="*60)
response = chat("Based on what you told me earlier, what are the key components of LangChain?")
print(f"\nFinal Answer: {response}")

Test 4: Memory - follow-up question

Final Answer: [{'text': 'Based on what I told you earlier, the key components of LangChain are:\n\n1. **Chains** - sequences of operations for processing\n2. **Agents** - autonomous decision-making components\n3. **Memory** - for maintaining context and history\n4. **Tools** - integrations with external services and APIs\n\nThese components work together to help developers build complex applications powered by Large Language Models.', 'type': 'text', 'index': 0}]


In [18]:
@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between units. Supports:
    - Temperature: celsius, fahrenheit, kelvin
    - Length: meters, feet, inches, kilometers, miles
    - Weight: kg, pounds, ounces
    """
    # YOUR CODE HERE
    # Example conversions:
    # celsius to fahrenheit: (value * 9/5) + 32
    # meters to feet: value * 3.28084
    # kg to pounds: value * 2.20462
    pass

result = convert_units.invoke({"value": 100, "from_unit": "celsius", "to_unit": "fahrenheit"})
print(result)

None


In [19]:
'''EXample:
 
You are a professional financial advisor assistant.
    
# YOUR INSTRUCTIONS HERE:
# - Define the persona
# - Set guidelines for responses
# - Add disclaimer requirements


# Create and test the financial advisor agent
'''

'EXample:\n\nYou are a professional financial advisor assistant.\n\n# YOUR INSTRUCTIONS HERE:\n# - Define the persona\n# - Set guidelines for responses\n# - Add disclaimer requirements\n\n\n# Create and test the financial advisor agent\n'

In [3]:
WEATHER_DATA = {
    "paris": "Sunny, 22°C",
    "tokyo": "Cloudy, 18°C",
    "new york": "Rainy, 15°C"
}

ATTRACTIONS = {
    "paris": ["Eiffel Tower", "Louvre Museum", "Notre-Dame"],
    "tokyo": ["Tokyo Tower", "Senso-ji Temple", "Shibuya Crossing"],
    "new york": ["Statue of Liberty", "Central Park", "Times Square"]
}

FLIGHT_DATA = {
    ("new york", "paris"): ["NY101 at 09:00 AM", "NY205 at 06:30 PM"],
    ("new york", "tokyo"): ["NY330 at 11:45 AM"],
    ("paris", "tokyo"): ["PA440 at 02:15 PM"],
    ("tokyo", "paris"): ["TK510 at 08:20 AM"],
    ("paris", "new york"): ["PA120 at 10:00 AM"],
    ("tokyo", "new york"): ["TK880 at 04:45 PM"]
}


@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    city_key = city.lower().strip()

    if city_key in WEATHER_DATA:
        return f"The current weather in {city.title()} is: {WEATHER_DATA[city_key]}"

    return f"Weather information is not available for {city}."


@tool
def search_flights(from_city: str, to_city: str, date: str) -> str:
    """Search for available flights."""
    from_key = from_city.lower().strip()
    to_key = to_city.lower().strip()

    flights = FLIGHT_DATA.get((from_key, to_key))

    if flights:
        flight_list = ", ".join(flights)
        return f"Available flights from {from_city.title()} to {to_city.title()} on {date}: {flight_list}"

    return f"No simulated flight data found from {from_city.title()} to {to_city.title()} on {date}."


@tool
def get_attractions(city: str) -> str:
    """Get popular tourist attractions in a city."""
    city_key = city.lower().strip()

    if city_key in ATTRACTIONS:
        attractions = ", ".join(ATTRACTIONS[city_key])
        return f"Popular attractions in {city.title()}: {attractions}"

    return f"Attraction information is not available for {city}."


travel_tools = [get_weather, search_flights, get_attractions]

travel_system_prompt = """You are a helpful travel planning assistant.

Your capabilities:
- Get current weather for supported cities
- Search for simulated flight options
- Recommend popular tourist attractions

Guidelines:
1. Use tools whenever the user asks about weather, flights, or attractions.
2. If a city is not in the simulated data, say that information is unavailable.
3. Give practical, concise travel advice.
4. Ask for missing details if the user wants flight information but has not provided departure city or date.
"""

travel_agent = create_agent(
    model=llm,
    tools=travel_tools,
    system_prompt=travel_system_prompt
)

print("Travel Planning Agent created!")


def travel_chat(user_input: str) -> str:
    """Send a message to the travel planning agent."""
    result = travel_agent.invoke({
        "messages": [HumanMessage(content=user_input)]
    })

    return result["messages"][-1].content


# Step 4: Test with: "I'm planning a trip to Paris next week. What's the weather like and what should I see?"
response = travel_chat(
    "I'm planning a trip to Paris next week. What's the weather like and what should I see?"
)

print(response)

Travel Planning Agent created!
[{'text': "Great! Here's what you need to know about your Paris trip:\n\n**Weather:**\nParis is currently sunny with a pleasant temperature of 22°C (about 72°F). Perfect weather for sightseeing! I'd recommend bringing light layers and comfortable walking shoes.\n\n**Must-See Attractions:**\n- **Eiffel Tower** - The iconic symbol of Paris. Visit during sunset for stunning views\n- **Louvre Museum** - The world's largest art museum, home to the Mona Lisa and countless masterpieces\n- **Notre-Dame** - The magnificent Gothic cathedral (note: it's currently undergoing restoration, but the exterior is still impressive)\n\n**Travel Tips:**\n- Book museum tickets in advance to skip long queues\n- Consider getting a Paris Museum Pass if you plan to visit multiple attractions\n- The city is very walkable, but the metro is efficient and affordable\n- Wear comfortable shoes—you'll be doing a lot of walking!\n\nWould you like help searching for flights to Paris? If so